#  BeautifulSoup Documentation Analytics System

**Course: PDS301m - Python for Data Science**

---

##  Project Overview

This project analyzes the official BeautifulSoup documentation to extract meaningful insights. BeautifulSoup is a popular Python library for web scraping.

###  Project Goals

1. **Collect** documentation pages from the web
2. **Parse** HTML content using BeautifulSoup
3. **Extract** structured data (sections, links, code examples)
4. **Analyze** the documentation using Python
5. **Visualize** key findings with intuitive charts
6. **Report** insights in a clear, accessible format

###  Data Sources

- **Input:** BeautifulSoup Official Documentation (https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- **Output:** CSV files, charts, and analytical reports

---

##  Project Architecture

| Component | File | Description |
|:---|:---|:---|
| Feature 1 | `collector.py` | Downloads HTML from website |
| Feature 2 | `parser.py` | Parses HTML with BeautifulSoup |
| Features 3-5 | `extractor.py` | Extracts Sections, Links, Code Examples |
| Feature 6 | `analyzer.py` | Analyzes data with Pandas |
| Feature 7 | `visualizer.py` | Creates visualizations |
| Feature 8 | `report_generator.py` | Generates final reports |

## 1️ Setup & Imports

First, let's import all necessary libraries.

In [ ]:
import os
import sys
import re
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

# Set style for better visualizations
plt.style.use('seaborn-v0_8-whitegrid')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', None)

print(' All libraries imported successfully!')

## 2️ Set Working Directory

Ensure we're working in the correct project directory.

In [ ]:
# Navigate to project root
if 'src' in os.listdir('.') and 'data' in os.listdir('.'):
    print(' Already in project root:', os.getcwd())
else:
    # Try to find project root
    paths_to_try = ['.', '..', '../..']
    for path in paths_to_try:
        if os.path.exists(os.path.join(path, 'src')) and os.path.exists(os.path.join(path, 'data')):
            os.chdir(path)
            print(' Changed to project root:', os.getcwd())
            break

sys.path.insert(0, os.getcwd())

# Create necessary directories
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('output/charts', exist_ok=True)

print(' Directory structure ready!')

---

## 3️ Feature 1: Web Page Collector 

Download the BeautifulSoup documentation from the official website.

In [ ]:
def collect_webpage(url='https://www.crummy.com/software/BeautifulSoup/bs4/doc/'):
    """
    Download HTML content from a URL and save to file.
    """
    print(f' Downloading: {url}')
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        html_content = response.text
        
        # Save to file
        output_path = 'data/raw/beautifulsoup_doc.html'
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(html_content)
        
        print(f' Saved {len(html_content):,} characters to {output_path}')
        return html_content
        
    except requests.RequestException as e:
        print(f' Error downloading: {e}')
        return None

html_content = collect_webpage()

## 4️ Feature 2: HTML Parser 

Parse the HTML file using BeautifulSoup to create a navigable tree structure.

In [ ]:
def parse_html(html_content):
    """
    Parse HTML content and return BeautifulSoup object.
    """
    if html_content is None:
        print(' No HTML content to parse')
        return None
    
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Extract page title
    title = soup.find('title')
    title_text = title.get_text().strip() if title else 'No title found'
    
    print(f' Page title: {title_text}')
    print(f' Parsed HTML successfully!')
    
    return soup

# Try to load from file first, or download
html_file = 'data/raw/beautifulsoup_doc.html'
if os.path.exists(html_file):
    with open(html_file, 'r', encoding='utf-8') as f:
        html_content = f.read()
    print(f' Loaded HTML from file: {len(html_content):,} characters')
else:
    html_content = collect_webpage()

soup = parse_html(html_content)

## 5️ Feature 3: Section Extractor 

Extract all documentation sections (headings) and compute their statistics.

In [ ]:
def clean_text(text):
    """Remove extra whitespace and special characters."""
    if text is None:
        return ''
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', str(text))
    # Remove pilcrow sign and other special characters
    text = re.sub(r'[\u00b6\u00a7\u00a0]', '', text)
    return text.strip()

def extract_sections(soup):
    """
    Extract all sections from the documentation.
    """
    HEADING_TAGS = ['h1', 'h2', 'h3']
    headings = soup.find_all(HEADING_TAGS)
    
    rows = []
    for idx, heading in enumerate(headings, start=1):
        # Get section title
        section_title = clean_text(heading.get_text(' ', strip=True))
        section_level = heading.name
        
        # Get content until next heading
        content = []
        for sibling in heading.find_next_siblings():
            if sibling.name in HEADING_TAGS:
                break
            content.append(sibling)
        
        # Calculate statistics
        text_parts = [clean_text(el.get_text(' ', strip=True)) for el in content]
        section_text = ' '.join(text_parts)
        word_count = len(section_text.split())
        code_block_count = sum(len(el.find_all(['pre', 'code'])) for el in content)
        link_count = sum(len(el.find_all('a')) for el in content)
        
        rows.append({
            'section_id': idx,
            'section_level': section_level,
            'section_title': section_title,
            'section_text': section_text[:500] + '...' if len(section_text) > 500 else section_text,
            'word_count': word_count,
            'code_block_count': code_block_count,
            'link_count': link_count
        })
    
    return pd.DataFrame(rows)

# Extract sections
df_sections = extract_sections(soup)
df_sections.to_csv('data/processed/sections.csv', index=False, encoding='utf-8-sig')

print(f' Extracted {len(df_sections)} sections')
print(f'\n Section Summary:')
print(df_sections[['section_id', 'section_level', 'section_title', 'word_count']].head(10))

## 6️ Feature 4: Link Extractor 

Extract all hyperlinks and classify them by type.

In [ ]:
def get_closest_section(element, headings):
    """Find the nearest preceding heading."""
    for heading in reversed(headings):
        if heading.get('id') == element.get('id'):
            continue
        if heading.sourceline < element.sourceline:
            return clean_text(heading.get_text(' ', strip=True))
    return 'Unknown'

def classify_link(href):
    """Classify link type based on URL."""
    if not href or href == '#' or href.startswith('javascript:'):
        return 'empty_or_invalid'
    elif href.startswith('#'):
        return 'internal_anchor'
    elif any(href.lower().endswith(ext) for ext in ['.png', '.jpg', '.jpeg', '.gif', '.svg']):
        return 'image_link'
    elif href.startswith(('http://', 'https://')):
        if 'beautifulsoup' in href.lower() or 'crummy' in href.lower():
            return 'documentation_link'
        return 'external_link'
    else:
        return 'documentation_link'

def extract_links(soup):
    """Extract all links from the documentation."""
    headings = soup.find_all(['h1', 'h2', 'h3'])
    rows = []
    
    for a_tag in soup.find_all('a'):
        href = a_tag.get('href', '').strip()
        link_text = clean_text(a_tag.get_text())
        section_title = get_closest_section(a_tag, headings)
        link_type = classify_link(href)
        
        rows.append({
            'link_text': link_text if link_text else '(empty)',
            'href': href,
            'link_type': link_type,
            'section_title': section_title
        })
    
    return pd.DataFrame(rows)

# Extract links
df_links = extract_links(soup)
df_links.to_csv('data/processed/links.csv', index=False, encoding='utf-8-sig')

print(f' Extracted {len(df_links)} links')
print(f'\n Link Type Distribution:')
print(df_links['link_type'].value_counts())

## 7️ Feature 5: Code Example Extractor 

Extract all Python code examples and analyze their content.

In [ ]:
def extract_code_examples(soup):
    """Extract code examples from <pre> tags."""
    headings = soup.find_all(['h1', 'h2', 'h3'])
    code_blocks = soup.find_all('pre')
    
    rows = []
    for idx, block in enumerate(code_blocks, start=1):
        code_text = block.get_text().strip()
        if not code_text:
            continue
        
        # Find closest section
        section_title = get_closest_section(block, headings)
        
        # Count lines
        line_count = len(code_text.split('\n'))
        
        rows.append({
            'example_id': f'ex_{idx}',
            'section_title': section_title,
            'code_text': code_text,
            'line_count': line_count,
            'contains_find_all': 'find_all' in code_text,
            'contains_find': 'find(' in code_text or '.find ' in code_text,
            'contains_select': 'select' in code_text,
            'contains_get_text': 'get_text' in code_text,
            'contains_requests': 'requests' in code_text
        })
    
    return pd.DataFrame(rows)

# Extract code examples
df_code = extract_code_examples(soup)
df_code.to_csv('data/processed/code_examples.csv', index=False, encoding='utf-8-sig')

print(f' Extracted {len(df_code)} code examples')
print(f'\n Code Examples by Section:')
print(df_code['section_title'].value_counts().head(5))

---

## 8️ Feature 6: Documentation Analytics 

Answer key analytical questions about the documentation.

In [ ]:
# Load extracted data
df_sections = pd.read_csv('data/processed/sections.csv')
df_links = pd.read_csv('data/processed/links.csv')
df_code = pd.read_csv('data/processed/code_examples.csv')

print('='*60)
print('          DOCUMENTATION ANALYTICS REPORT')
print('='*60)

In [ ]:
print('\n' + '='*60)
print('              REQUIRED QUESTIONS (Q1-Q8)')
print('='*60)

# Q1: How many sections?
print('\n Q1: How many sections are in the documentation?')
print(f'   ➜ Answer: {len(df_sections)} sections found')
print(f'   • h1 headings (main topics): {len(df_sections[df_sections["section_level"]=="h1"])}')
print(f'   • h2 headings (subtopics): {len(df_sections[df_sections["section_level"]=="h2"])}')
print(f'   • h3 headings (details): {len(df_sections[df_sections["section_level"]=="h3"])}')

In [ ]:
# Q2: Which section has highest word count?
print('\n Q2: Which section has the highest word count?')
max_word_section = df_sections.loc[df_sections['word_count'].idxmax()]
print(f'   ➜ Answer: "{max_word_section["section_title"]}"')
print(f'   • Word count: {max_word_section["word_count"]:,} words')

In [ ]:
# Q3: Which section has most code examples?
print('\n Q3: Which section contains the most code examples?')
code_by_section = df_code['section_title'].value_counts()
top_code_section = code_by_section.index[0]
top_code_count = code_by_section.iloc[0]
print(f'   ➜ Answer: "{top_code_section}"')
print(f'   • Code blocks: {top_code_count}')

In [ ]:
# Q4: Which section has most links?
print('\n Q4: Which section contains the most links?')
max_link_section = df_sections.loc[df_sections['link_count'].idxmax()]
print(f'   ➜ Answer: "{max_link_section["section_title"]}"')
print(f'   • Links: {max_link_section["link_count"]}')

In [ ]:
# Q5: Top 10 technical keywords
print('\n Q5: What are the top 10 most frequent technical keywords?')
from collections import Counter

all_words = ' '.join(df_sections['section_text'].dropna().astype(str)).split()
# Filter common stopwords
stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
             'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'been',
             'be', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
             'should', 'may', 'might', 'must', 'shall', 'can', 'this', 'that', 'these',
             'those', 'it', 'its', 'they', 'them', 'their', 'you', 'your', 'we', 'our',
             'i', 'my', 'me', 'if', 'so', 'not', 'no', 'any', 'all', 'each', 'other',
             'than', 'then', 'when', 'where', 'which', 'who', 'what', 'how', 'why',
             'here', 'there', 'out', 'up', 'down', 'into', 'through', 'over', 'under'}
filtered_words = [w.lower() for w in all_words if w.lower() not in stopwords and len(w) > 3]
word_counts = Counter(filtered_words).most_common(10)

for i, (word, count) in enumerate(word_counts, 1):
    print(f'   {i:2}. {word}: {count} times')

In [ ]:
# Q6: Internal vs External links
print('\n Q6: How many internal and external links exist?')
link_type_counts = df_links['link_type'].value_counts()

internal = link_type_counts.get('internal_anchor', 0) + link_type_counts.get('documentation_link', 0)
external = link_type_counts.get('external_link', 0)

print(f'   ➜ Internal links: {internal} (anchor + documentation)')
print(f'   ➜ External links: {external}')
print(f'\n   Breakdown by type:')
for link_type, count in link_type_counts.items():
    print(f'   • {link_type}: {count}')

In [ ]:
# Q7: Code examples using find_all()
print('\n Q7: How many code examples use find_all()?')
find_all_count = df_code['contains_find_all'].sum()
print(f'   ➜ Answer: {find_all_count} examples')

# Q8: Code examples using get_text()
print('\n Q8: How many code examples use get_text()?')
get_text_count = df_code['contains_get_text'].sum()
print(f'   ➜ Answer: {get_text_count} examples')

In [ ]:
print('\n' + '='*60)
print('              ADDITIONAL QUESTIONS (Q9-Q10)')
print('='*60)

# Q9: Average lines and longest example
print('\n Q9: Average code example length & longest example?')
avg_lines = df_code['line_count'].mean()
longest = df_code.loc[df_code['line_count'].idxmax()]
print(f'   ➜ Average lines per example: {avg_lines:.2f} lines')
print(f'   ➜ Longest example: "{longest["section_title"]}"')
print(f'     ({longest["line_count"]} lines)')

# Q10: Statistics by heading level
print('\n Q10: Statistics by heading level (h1, h2, h3)')
level_stats = df_sections.groupby('section_level').agg({
    'word_count': ['mean', 'sum'],
    'code_block_count': ['mean', 'sum'],
    'link_count': ['mean', 'sum']
}).round(2)
print(level_stats)

---

## 9️ Feature 7: Data Visualization 

Create intuitive charts that anyone can understand.

In [ ]:
# ============================================
# CHART SETUP: Beautiful Styling Configuration
# ============================================
import matplotlib.pyplot as plt
import matplotlib

# Set font and styling
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

# Beautiful, easy-to-distinguish color palette
COLORS = {
    'primary':   '#1E88E5',   # Blue
    'secondary': '#E91E63',   # Pink/Red
    'accent':    '#FB8C00',   # Orange
    'success':   '#43A047',   # Green
    'info':      '#00ACC1',   # Cyan
    'purple':    '#8E24AA',   # Purple
    'teal':      '#00897B',   # Teal
    'amber':     '#FFB300',   # Amber
    'indigo':    '#3949AB',   # Indigo
    'brown':     '#6D4C41',   # Brown
}

print("Chart styling configured with easy-to-distinguish colors!")


In [ ]:
# ============================================
# CHART 1: Top Sections by Word Count
# ============================================
import pandas as pd

top_sections = df_sections.nlargest(10, 'word_count')[['section_title', 'word_count']].copy()
top_sections = top_sections.sort_values('word_count', ascending=True)

# Shorten labels to max 30 chars
top_sections['short_title'] = top_sections['section_title'].apply(
    lambda x: x[:30] + '...' if len(x) > 30 else x
)

fig, ax = plt.subplots(figsize=(13, 8))
fig.patch.set_facecolor('white')

# Color gradient - darker for larger values
colors = [COLORS['primary'] if i % 2 == 0 else '#64B5F6' 
          for i in range(len(top_sections))]

bars = ax.barh(top_sections['short_title'], top_sections['word_count'], 
               color=colors, edgecolor='white', linewidth=1.5, height=0.7)

# Add value labels on the right of bars
max_val = top_sections['word_count'].max()
for bar, val in zip(bars, top_sections['word_count']):
    ax.text(val + max_val * 0.02, bar.get_y() + bar.get_height()/2, 
            f'{val:,}', va='center', fontsize=10, fontweight='bold', color='#333333')

ax.set_xlabel('Word Count', fontsize=13, fontweight='bold', labelpad=10)
ax.set_title('Top 10 Sections by Word Count
How Much Content Does Each Section Have?', 
             fontsize=15, fontweight='bold', pad=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.set_xlim(0, max_val * 1.18)
ax.tick_params(axis='y', labelsize=10)
ax.tick_params(axis='x', labelsize=10)
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('output/charts/chart_01_word_count.png', dpi=150, bbox_inches='tight', 
            facecolor='white')
plt.show()

print('\n[OK] Chart 1 saved: output/charts/chart_01_word_count.png')


In [ ]:
# ============================================
# CHART 2: Code Examples by Section
# ============================================
import pandas as pd

df_code = pd.read_csv('data/processed/code_examples.csv')
code_by_section = df_code['section_title'].value_counts()

# Group small sections (< 3 examples) into 'Others'
threshold = 3
main_sections = code_by_section[code_by_section >= threshold]
other_count = code_by_section[code_by_section < threshold].sum()

if other_count > 0:
    labels = list(main_sections.index) + ['Others']
    values = list(main_sections.values) + [other_count]
else:
    labels = list(main_sections.index)
    values = list(main_sections.values)

# Sort ascending for horizontal bar (largest at top)
sorted_pairs = sorted(zip(labels, values), key=lambda x: x[1], ascending=True)
labels = [p[0] for p in sorted_pairs]
values = [p[1] for p in sorted_pairs]

# Shorten labels to max 32 chars
short_labels = [l[:32] + '...' if len(l) > 32 else l for l in labels]

fig, ax = plt.subplots(figsize=(13, max(7, len(labels) * 0.7)))
fig.patch.set_facecolor('white')

# Color gradient
import numpy as np
n = len(values)
colors_grad = [COLORS['primary'], COLORS['secondary'], COLORS['accent'], 
               COLORS['success'], COLORS['info'], COLORS['purple'],
               COLORS['teal'], COLORS['amber'], COLORS['indigo'], COLORS['brown']]

y_pos = np.arange(len(short_labels))
bars = ax.barh(y_pos, values, color=colors_grad[:n], 
               edgecolor='white', linewidth=1.5, height=0.7)

# Add count + percentage labels at end of bars
total = sum(values)
for bar, val, label in zip(bars, values, short_labels):
    pct = val / total * 100
    ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height()/2,
            f'{val} ({pct:.1f}%)', va='center', ha='left',
            fontsize=10, fontweight='bold', color='#333333')

ax.set_yticks(y_pos)
ax.set_yticklabels(short_labels, fontsize=10)
ax.set_xlabel('Number of Code Examples', fontsize=13, fontweight='bold', labelpad=10)
ax.set_title('Code Examples by Section
(How Are Code Examples Distributed?)',
             fontsize=15, fontweight='bold', pad=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.set_xlim(0, max(values) * 1.25)
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('output/charts/chart_02_code_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()

print('\n[OK] Chart 2 saved: output/charts/chart_02_code_distribution.png')


In [ ]:
# ============================================
# CHART 3: Link Type Distribution
# ============================================
import pandas as pd

df_links = pd.read_csv('data/processed/links.csv')
link_counts = df_links['link_type'].value_counts()

# Sort ascending so largest bar is on top
link_counts = link_counts.sort_values(ascending=True)

# Better color palette for link types
link_colors = {
    'internal_anchor':     COLORS['success'],
    'documentation_link':  COLORS['primary'],
    'external_link':       COLORS['accent'],
    'image_link':          COLORS['purple'],
    'empty_or_invalid':   COLORS['brown'],
}
colors = [link_colors.get(lt, '#9E9E9E') for lt in link_counts.index]

# Make labels more readable (replace underscores with spaces, title case)
pretty_labels = [lt.replace('_', ' ').title() for lt in link_counts.index]

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('white')

y_pos = np.arange(len(link_counts))
bars = ax.barh(y_pos, link_counts.values, color=colors,
               edgecolor='white', linewidth=2, height=0.65)

# Add count + percentage labels at end of bars
total = len(df_links)
for bar, count in zip(bars, link_counts.values):
    pct = count / total * 100
    ax.text(bar.get_width() + max(link_counts.values) * 0.01,
            bar.get_y() + bar.get_height()/2,
            f'{count:,}  ({pct:.1f}%)',
            va='center', ha='left',
            fontsize=11, fontweight='bold', color='#333333')

ax.set_yticks(y_pos)
ax.set_yticklabels(pretty_labels, fontsize=12, fontweight='medium')
ax.set_xlabel('Number of Links', fontsize=13, fontweight='bold', labelpad=10)
ax.set_title(f'Link Types Distribution\n(Total: {total:,} links)',
             fontsize=15, fontweight='bold', pad=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.set_xlim(0, max(link_counts.values) * 1.25)
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('output/charts/chart_03_link_types.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()

print('\n[OK] Chart 3 saved: output/charts/chart_03_link_types.png')


In [ ]:
# ============================================
# CHART 4: Code Example Length Distribution
# ============================================
import pandas as pd

df_code = pd.read_csv('data/processed/code_examples.csv')

fig, axes = plt.subplots(1, 2, figsize=(15, 6), 
                          gridspec_kw={'width_ratios': [2, 1]})

# --- Left: Histogram ---
ax1 = axes[0]
n, bins, patches = ax1.hist(df_code['line_count'], bins=15, 
                              color=COLORS['primary'], 
                              edgecolor='white', linewidth=1.5, alpha=0.85)

mean_val = df_code['line_count'].mean()
median_val = df_code['line_count'].median()
mode_val = df_code['line_count'].mode()[0]

# Add vertical lines for mean and median
ax1.axvline(mean_val, color='#E53935', linestyle='--', linewidth=2.5,
            label=f'Mean = {mean_val:.1f}')
ax1.axvline(median_val, color='#1E88E5', linestyle='-.', linewidth=2.5,
            label=f'Median = {median_val:.1f}')

ax1.set_xlabel('Number of Lines', fontsize=13, fontweight='bold')
ax1.set_ylabel('Frequency (Number of Examples)', fontsize=13, fontweight='bold')
ax1.set_title('Code Example Length Distribution\nHow Long Are Code Examples?', 
              fontsize=14, fontweight='bold', pad=15)
ax1.legend(fontsize=11, frameon=True, fancybox=True, shadow=True)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.grid(axis='y', alpha=0.3, linestyle='--')
ax1.set_axisbelow(True)

# --- Right: Statistics box ---
ax2 = axes[1]
ax2.axis('off')

# Create a styled stats box
stats_data = [
    ('Total Examples', f'{len(df_code):,}'),
    ('Average Length', f'{mean_val:.1f} lines'),
    ('Median Length', f'{median_val:.1f} lines'),
    ('Most Common', f'{mode_val} line(s)'),
    ('Shortest', f'{df_code["line_count"].min()} line(s)'),
    ('Longest', f'{df_code["line_count"].max()} lines'),
    ('Std Deviation', f'{df_code["line_count"].std():.1f}'),
]

# Draw box background
bbox_props = dict(boxstyle='round,pad=0.6', facecolor='#F5F5F5', 
                  edgecolor='#BDBDBD', linewidth=2)
ax2.text(0.5, 0.95, 'Statistics Summary', fontsize=14, fontweight='bold',
         ha='center', va='top', transform=ax2.transAxes, color='#333333')
ax2.text(0.5, 0.82, '─' * 22, fontsize=11, ha='center', va='top', 
         transform=ax2.transAxes, color='#888888', family='monospace')

y_start = 0.68
for label, value in stats_data:
    ax2.text(0.08, y_start, label + ':', fontsize=12, ha='left', va='center',
             transform=ax2.transAxes, color='#555555', fontweight='medium')
    ax2.text(0.92, y_start, value, fontsize=12, ha='right', va='center',
             transform=ax2.transAxes, color='#333333', fontweight='bold',
             family='monospace')
    y_start -= 0.11

ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('output/charts/chart_04_code_length.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()

print('\n[OK] Chart 4 saved: output/charts/chart_04_code_length.png')


In [ ]:
# ============================================
# CHART 5: BeautifulSoup API Usage
# ============================================
import pandas as pd

df_code = pd.read_csv('data/processed/code_examples.csv')

api_usage = {
    'find_all()':   df_code['contains_find_all'].sum(),
    'find()':       df_code['contains_find'].sum(),
    'select()':     df_code['contains_select'].sum(),
    'get_text()':   df_code['contains_get_text'].sum(),
    'requests':     df_code['contains_requests'].sum()
}

# Sort by value descending
api_usage = dict(sorted(api_usage.items(), key=lambda x: x[1], reverse=True))

# Color palette
api_colors = [COLORS['primary'], COLORS['secondary'], COLORS['accent'],
              COLORS['success'], COLORS['info']]

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('white')

x_pos = np.arange(len(api_usage))
bars = ax.bar(x_pos, list(api_usage.values()), color=api_colors,
              edgecolor='white', linewidth=2, width=0.6)

# Add value labels on top of bars
max_val = max(api_usage.values())
for bar, val in zip(bars, api_usage.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max_val * 0.015,
            f'{int(val)}', ha='center', va='bottom',
            fontsize=13, fontweight='bold', color='#333333')

ax.set_xticks(x_pos)
ax.set_xticklabels(list(api_usage.keys()), fontsize=13, fontweight='medium')
ax.set_ylabel('Number of Code Examples', fontsize=13, fontweight='bold', labelpad=10)
ax.set_title('BeautifulSoup API Methods Used in Documentation\n'
             'Which Functions Are Demonstrated Most?', 
             fontsize=15, fontweight='bold', pad=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.set_ylim(0, max_val * 1.18)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)
ax.tick_params(axis='y', labelsize=11)

plt.tight_layout()
plt.savefig('output/charts/chart_05_api_usage.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()

print('\n[OK] Chart 5 saved: output/charts/chart_05_api_usage.png')


---

## 10 Feature 8: Summary Report 

Key findings and insights from the analysis.

In [ ]:
print('\n' + '='*70)
print('                     FINAL ANALYSIS SUMMARY')
print('='*70)

print('''
    ┌─────────────────────────────────────────────────────────────┐
    │                    KEY FINDINGS                             │
    └─────────────────────────────────────────────────────────────┘
    
     DOCUMENTATION STRUCTURE
    • Total sections: 113 (19 main topics, 52 subtopics, 42 details)
    • Total words: 40,879 words across all sections
    • Total links: 504 hyperlinks
    • Total code examples: 220 blocks
    
    TOP PERFORMING SECTIONS
    • "Searching the tree" - Most content (4,687 words, 208 code blocks)
    • "Table of Contents" - Most links (127 links)
    • "Parsing only part of a document" - Longest code example (37 lines)
    
    CODE EXAMPLE INSIGHTS
    • Average code length: 6.34 lines
    • Most used method: find_all() appears in 41 examples
    • Second most used: find() appears in many examples
    
    LINK ANALYSIS
    • Internal links: 485 (96.2%)
    • External links: 17 (3.4%)
    • Most links are anchor links (#section-name)
    
    ┌─────────────────────────────────────────────────────────────┐
    │                    CONCLUSIONS                               │
    └─────────────────────────────────────────────────────────────┘
    
    1. The documentation is comprehensive with 113 sections
    2. "Searching the tree" is the most detailed section
    3. Code examples are concise (avg 6 lines)
    4. Documentation is well-linked internally
    5. find_all() is the most demonstrated method
''')

print('='*70)
print('                 Analysis completed successfully!')
print('='*70)